# 06 — Model Analysis FINAL | Eye Disease Detection

This notebook reproduces the **exact test pipeline used in the EfficientNetV2 training notebook** and then performs model analysis.

Important:
- Uses the same `test.csv`
- Uses the same `load_image()` logic
- Uses the same `make_dataset()` logic
- Keeps pixels in **0–255**
- Uses the saved `efficientnetv2_finetuned_best.keras`
- Does NOT manually rebuild `X_test` with PIL/load_img
- Does NOT retrain the model

Run from Step 1 to the end.


In [1]:
# STEP 1 — Imports

import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
)

warnings.filterwarnings("ignore")

print("TensorFlow:", tf.__version__)
print("Keras:", getattr(keras, "__version__", "unknown"))


TensorFlow: 2.21.0
Keras: 3.15.1


In [2]:
# STEP 2 — Project paths

PROJECT_ROOT = Path(r"D:/Practice Projects/Disease Detection")

SPLIT_DIR = PROJECT_ROOT / "preprocessing" / "splits"
MODEL_DIR = PROJECT_ROOT / "model"
REPORT_DIR = PROJECT_ROOT / "reports" / "model_analysis"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TEST_CSV = SPLIT_DIR / "test.csv"
FINETUNED_MODEL_PATH = MODEL_DIR / "efficientnetv2_finetuned_best.keras"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TEST_CSV:", TEST_CSV)
print("MODEL:", FINETUNED_MODEL_PATH)

if not TEST_CSV.exists():
    raise FileNotFoundError(f"Test CSV not found: {TEST_CSV}")

if not FINETUNED_MODEL_PATH.exists():
    matches = list(MODEL_DIR.rglob("*finetuned*best*.keras"))
    if matches:
        FINETUNED_MODEL_PATH = matches[0]
    else:
        raise FileNotFoundError("Fine-tuned .keras checkpoint was not found.")


PROJECT_ROOT: D:\Practice Projects\Disease Detection
TEST_CSV: D:\Practice Projects\Disease Detection\preprocessing\splits\test.csv
MODEL: D:\Practice Projects\Disease Detection\model\efficientnetv2_finetuned_best.keras


In [3]:
# STEP 3 — Reproducibility and constants
# Same values used by the training pipeline

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
NUM_CLASSES = 5
AUTOTUNE = tf.data.AUTOTUNE

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("IMG_SIZE:", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("NUM_CLASSES:", NUM_CLASSES)


IMG_SIZE: (224, 224)
BATCH_SIZE: 16
NUM_CLASSES: 5


In [4]:
# STEP 4 — Load test CSV

test_df = pd.read_csv(TEST_CSV)

print("Test rows:", len(test_df))
print("Columns:", list(test_df.columns))
display(test_df.head())


Test rows: 270
Columns: ['image_path', 'label']


,image_path,label
0,../dataset/Messidor-2+EyePac_Balanced\3\200604...,3
1,../dataset/Messidor-2+EyePac_Balanced\0\200511...,0
2,../dataset/Messidor-2+EyePac_Balanced\1\200512...,1
3,../dataset/Messidor-2+EyePac_Balanced\1\200604...,1
4,../dataset/Messidor-2+EyePac_Balanced\0\200604...,0


In [5]:
# STEP 5 — Resolve label and image-path columns

def find_column(df, candidates):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    return None

IMAGE_COL = find_column(
    test_df,
    ["image_path", "filepath", "file_path", "path", "image", "filename"]
)

LABEL_COL = find_column(
    test_df,
    ["label", "class", "class_id", "target", "disease", "y"]
)

if IMAGE_COL is None:
    raise ValueError(f"Image-path column not found. Columns: {list(test_df.columns)}")

if LABEL_COL is None:
    raise ValueError(f"Label column not found. Columns: {list(test_df.columns)}")

print("IMAGE_COL:", IMAGE_COL)
print("LABEL_COL:", LABEL_COL)


IMAGE_COL: image_path
LABEL_COL: label


In [ ]:
# STEP 6 — Resolve paths exactly for the dataset

def resolve_path(value):
    p = Path(str(value))

    if p.is_absolute() and p.exists():
        return str(p)

    candidates = [
        PROJECT_ROOT / str(value),
        PROJECT_ROOT / "dataset" / str(value),
    ]

    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    # Preserve the training-project convention if the CSV already contains
    # a path that will be resolved by the operating system.
    return str(p)

test_df = test_df.copy()
test_df["resolved_path"] = test_df[IMAGE_COL].apply(resolve_path)

missing_mask = ~test_df["resolved_path"].apply(lambda p: Path(p).exists())

print("Total test images:", len(test_df))
print("Missing images:", int(missing_mask.sum()))

if missing_mask.any():
    display(test_df.loc[missing_mask, [IMAGE_COL, "resolved_path", LABEL_COL]].head(20))
    raise FileNotFoundError("Some test image paths do not exist.")

print("\nRaw label distribution:")
print(test_df[LABEL_COL].value_counts().sort_index())


In [ ]:
# STEP 7 — Convert labels to integer class IDs

def label_to_int(value):
    s = str(value).strip()

    if s.lower().startswith("class "):
        s = s[6:].strip()

    try:
        return int(float(s))
    except Exception:
        raise ValueError(f"Cannot convert label to class ID: {value}")

test_df["analysis_label"] = test_df[LABEL_COL].apply(label_to_int).astype(np.int32)

if not set(test_df["analysis_label"].unique()).issubset(set(range(NUM_CLASSES))):
    raise ValueError(
        f"Unexpected labels found: {sorted(test_df['analysis_label'].unique())}"
    )

print("Integer label distribution:")
print(test_df["analysis_label"].value_counts().sort_index())


In [ ]:
# STEP 8 — Exact training-notebook image loader

def load_image(path, label):
    image_bytes = tf.io.read_file(path)

    image = tf.image.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        IMG_SIZE,
        method=tf.image.ResizeMethod.BILINEAR
    )

    # IMPORTANT:
    # Keep 0–255 values.
    # EfficientNetV2 built-in preprocessing handles normalization.
    image = tf.cast(image, tf.float32)

    label = tf.cast(label, tf.int32)

    return image, label

print("Exact training image loader created.")


In [ ]:
# STEP 9 — Exact training-notebook test dataset creation

def make_test_dataset(df):
    paths = df["resolved_path"].values
    labels = df["analysis_label"].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    ds = ds.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(
        BATCH_SIZE,
        drop_remainder=False
    )

    ds = ds.prefetch(AUTOTUNE)

    return ds

test_ds = make_test_dataset(test_df)

print(test_ds)


In [ ]:
# STEP 10 — Verify test pipeline

x_batch, y_batch = next(iter(test_ds))

print("Batch shape:", x_batch.shape)
print("Label shape:", y_batch.shape)
print("Pixel min:", float(tf.reduce_min(x_batch)))
print("Pixel max:", float(tf.reduce_max(x_batch)))
print("Pixel mean:", float(tf.reduce_mean(x_batch)))

assert x_batch.shape[1:] == (224, 224, 3)
assert float(tf.reduce_min(x_batch)) >= 0
assert float(tf.reduce_max(x_batch)) <= 255.0 + 1e-3


In [ ]:
# STEP 11 — Load the saved fine-tuned model

final_model = keras.models.load_model(FINETUNED_MODEL_PATH)

print("Loaded model successfully:")
print(FINETUNED_MODEL_PATH)
print("Input shape:", final_model.input_shape)
print("Output shape:", final_model.output_shape)


In [ ]:
# STEP 12 — Reproduce the training notebook evaluation

test_loss, test_accuracy = final_model.evaluate(
    test_ds,
    verbose=1
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy (%): {test_accuracy * 100:.2f}%")

if len(test_df) != 270:
    print("WARNING: Expected 270 test images, but found:", len(test_df))


In [ ]:
# STEP 13 — Generate predictions from the SAME test_ds

pred_probs = final_model.predict(
    test_ds,
    verbose=1
)

pred_labels = np.argmax(
    pred_probs,
    axis=1
).astype(np.int32)

true_labels = np.concatenate(
    [labels.numpy() for _, labels in test_ds],
    axis=0
).astype(np.int32)

print("True labels shape:", true_labels.shape)
print("Predicted labels shape:", pred_labels.shape)
print("Prediction probabilities shape:", pred_probs.shape)

assert len(true_labels) == len(pred_labels)
assert pred_probs.shape[1] == NUM_CLASSES


In [ ]:
# STEP 14 — Cross-check accuracy

accuracy = accuracy_score(
    true_labels,
    pred_labels
)

correct = int(np.sum(true_labels == pred_labels))
incorrect = int(np.sum(true_labels != pred_labels))

print("Total Test Images:", len(true_labels))
print("Correct Predictions:", correct)
print("Incorrect Predictions:", incorrect)
print(f"Accuracy from predictions: {accuracy:.4f}")
print(f"Accuracy from predictions (%): {accuracy * 100:.2f}%")
print(f"Keras evaluate accuracy (%): {test_accuracy * 100:.2f}%")

assert abs(accuracy - test_accuracy) < 1e-6, (
    "Prediction accuracy does not match model.evaluate(). "
    "This indicates a test-pipeline mismatch."
)

print("PASS: Prediction accuracy matches model.evaluate().")


In [ ]:
# STEP 15 — Classification report

class_names = [f"Class {i}" for i in range(NUM_CLASSES)]

classification_text = classification_report(
    true_labels,
    pred_labels,
    labels=list(range(NUM_CLASSES)),
    target_names=class_names,
    zero_division=0
)

print(classification_text)

report_dict = classification_report(
    true_labels,
    pred_labels,
    labels=list(range(NUM_CLASSES)),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).T

report_path = REPORT_DIR / "classification_report.csv"
report_df.to_csv(report_path)

print("Saved:", report_path)


In [ ]:
# STEP 16 — Confusion matrix

cm = confusion_matrix(
    true_labels,
    pred_labels,
    labels=list(range(NUM_CLASSES))
)

print("Confusion Matrix:")
print(cm)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

cm_df.to_csv(REPORT_DIR / "confusion_matrix.csv")

fig, ax = plt.subplots(figsize=(7, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

plt.title("EfficientNetV2 Confusion Matrix")
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "confusion_matrix.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close()


In [ ]:
# STEP 17 — Per-class analysis

per_class_rows = []

for cls in range(NUM_CLASSES):
    total = int(np.sum(true_labels == cls))
    correct_cls = int(np.sum(
        (true_labels == cls) &
        (pred_labels == cls)
    ))

    errors_cls = total - correct_cls

    per_class_rows.append({
        "class": f"Class {cls}",
        "total": total,
        "correct": correct_cls,
        "errors": errors_cls,
        "accuracy": correct_cls / total if total else 0.0
    })

per_class_df = pd.DataFrame(per_class_rows)

display(per_class_df)

per_class_df.to_csv(
    REPORT_DIR / "per_class_analysis.csv",
    index=False
)


In [ ]:
# STEP 18 — Misclassification analysis

mis_rows = []

for true_cls in range(NUM_CLASSES):
    for predicted_cls in range(NUM_CLASSES):

        if true_cls == predicted_cls:
            continue

        count = int(np.sum(
            (true_labels == true_cls) &
            (pred_labels == predicted_cls)
        ))

        if count > 0:
            mis_rows.append({
                "true_class": true_cls,
                "predicted_class": predicted_cls,
                "count": count
            })

mis_df = pd.DataFrame(mis_rows)

if len(mis_df):
    mis_df = mis_df.sort_values(
        "count",
        ascending=False
    ).reset_index(drop=True)

    display(mis_df)

    mis_df.to_csv(
        REPORT_DIR / "misclassification_analysis.csv",
        index=False
    )
else:
    print("No misclassifications found.")


In [ ]:
# STEP 19 — Complete prediction table

confidence = np.max(
    pred_probs,
    axis=1
)

prediction_df = pd.DataFrame({
    "image_path": test_df[IMAGE_COL].astype(str),
    "true_class": true_labels,
    "predicted_class": pred_labels,
    "confidence": confidence,
    "correct": true_labels == pred_labels
})

for cls in range(NUM_CLASSES):
    prediction_df[f"prob_class_{cls}"] = pred_probs[:, cls]

prediction_path = REPORT_DIR / "test_predictions.csv"

prediction_df.to_csv(
    prediction_path,
    index=False
)

print("Saved:", prediction_path)

display(prediction_df.head(10))


In [ ]:
# STEP 20 — Confidence analysis

highest_idx = int(np.argmax(confidence))
lowest_idx = int(np.argmin(confidence))

print("=" * 55)
print("HIGHEST CONFIDENCE")
print("=" * 55)

print("Image:", prediction_df.loc[highest_idx, "image_path"])
print("True Class:", int(prediction_df.loc[highest_idx, "true_class"]))
print("Predicted Class:", int(prediction_df.loc[highest_idx, "predicted_class"]))
print("Confidence:", f"{confidence[highest_idx] * 100:.2f}%")

print("\n" + "=" * 55)
print("LOWEST CONFIDENCE")
print("=" * 55)

print("Image:", prediction_df.loc[lowest_idx, "image_path"])
print("True Class:", int(prediction_df.loc[lowest_idx, "true_class"]))
print("Predicted Class:", int(prediction_df.loc[lowest_idx, "predicted_class"]))
print("Confidence:", f"{confidence[lowest_idx] * 100:.2f}%")

print("\nAverage Confidence:", f"{confidence.mean() * 100:.2f}%")


In [ ]:
# STEP 21 — Correct vs incorrect confidence

correct_conf = confidence[true_labels == pred_labels]
incorrect_conf = confidence[true_labels != pred_labels]

print("Correct predictions:", len(correct_conf))
print("Incorrect predictions:", len(incorrect_conf))

if len(correct_conf):
    print(
        "Average correct confidence:",
        f"{correct_conf.mean() * 100:.2f}%"
    )

if len(incorrect_conf):
    print(
        "Average incorrect confidence:",
        f"{incorrect_conf.mean() * 100:.2f}%"
    )


In [ ]:
# STEP 22 — Confidence distribution

plt.figure(figsize=(8, 5))

plt.hist(
    confidence,
    bins=10,
    edgecolor="black"
)

plt.xlabel("Prediction Confidence")
plt.ylabel("Number of Images")
plt.title("Prediction Confidence Distribution")
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "confidence_distribution.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close()


In [ ]:
# STEP 23 — True vs predicted class distribution

true_counts = pd.Series(true_labels).value_counts().sort_index()
pred_counts = pd.Series(pred_labels).value_counts().sort_index()

x = np.arange(NUM_CLASSES)
width = 0.35

plt.figure(figsize=(9, 5))

plt.bar(
    x - width / 2,
    [true_counts.get(i, 0) for i in range(NUM_CLASSES)],
    width,
    label="True"
)

plt.bar(
    x + width / 2,
    [pred_counts.get(i, 0) for i in range(NUM_CLASSES)],
    width,
    label="Predicted"
)

plt.xticks(x, class_names)
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.title("True vs Predicted Class Distribution")
plt.legend()
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "true_vs_predicted_distribution.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("True distribution:")
print(true_counts)

print("\nPredicted distribution:")
print(pred_counts)


In [ ]:
# STEP 24 — Lowest-confidence predictions

lowest_conf_df = prediction_df.sort_values(
    "confidence",
    ascending=True
).head(20)

display(
    lowest_conf_df[
        [
            "image_path",
            "true_class",
            "predicted_class",
            "confidence",
            "correct"
        ]
    ]
)

lowest_conf_df.to_csv(
    REPORT_DIR / "lowest_confidence_predictions.csv",
    index=False
)


In [ ]:
# STEP 25 — Final summary

summary = pd.DataFrame([{
    "total_test_images": len(true_labels),
    "correct_predictions": correct,
    "incorrect_predictions": incorrect,
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "prediction_accuracy": float(accuracy),
    "average_confidence": float(confidence.mean()),
    "highest_confidence": float(confidence.max()),
    "lowest_confidence": float(confidence.min())
}])

summary_path = REPORT_DIR / "model_analysis_summary.csv"

summary.to_csv(
    summary_path,
    index=False
)

print("=" * 60)
print("MODEL ANALYSIS COMPLETED")
print("=" * 60)
print("Total Test Images:", len(true_labels))
print("Correct Predictions:", correct)
print("Incorrect Predictions:", incorrect)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Average Confidence: {confidence.mean() * 100:.2f}%")
print(f"Highest Confidence: {confidence.max() * 100:.2f}%")
print(f"Lowest Confidence: {confidence.min() * 100:.2f}%")
print("\nAll reports saved to:")
print(REPORT_DIR)
print("\nSummary saved:", summary_path)
